# Chess Tutor Web App – Codebase Walkthrough

## Overview

This notebook provides a detailed walkthrough of the project’s architecture, file structure, and core logic. The system is designed with a strong separation of concerns, dividing responsibilities across frontend, orchestration backend, and compute backends.

The system is composed of four major components:

- `public/` → Frontend (UI and client-side interaction)
- `src/` → Authoritative backend (game state + orchestration)
- `engine_backend/` → Engine evaluation (Stockfish-based analysis)
- `playerbot_backend/` → Human-like bot move generation

The backend (`src/`) is the **single source of truth**, while the Python services act as **compute workers**.

---

## High-Level File Structure
```js
project_root/
│
├── public/ // Frontend
│
├── src/ // Backend (authoritative orchestrator)
│ ├── controllers/ // Request orchestration logic
│ ├── game/ // Game state + status
│ ├── routes/ // API endpoints
│ └── services/ // External service interfaces
│
├── engine_backend/ // Engine evaluation service
├── playerbot_backend/ // Human-like bot service
│
└── server.js
```


This structure ensures clean separation between:
- UI
- State management
- Request orchestration
- Heavy computation

---

## Frontend (`public/`)

The frontend is responsible for all user interaction and rendering.

### Responsibilities

- Render chess board and UI panels
- Capture user actions (moves, toggles, settings)
- Send API requests to backend
- Display:
  - Updated board state
  - Move history
  - Engine feedback
  - Tutor responses

### Key Principle

The frontend is **stateless with respect to game rules**. It does not validate moves or compute outcomes—it relies entirely on the backend.

---

## Backend (`src/`) – Authoritative Orchestrator

The `src/` directory is the core of the system. It manages all state and coordinates every subsystem.

---

### Controllers (`controllers/`)

Controllers act as the **entry point for logic execution**. Each controller corresponds to a domain of responsibility:

- `board` → Board state queries and updates
- `move` → Move validation and execution
- `engine` → Engine evaluation requests
- `clock` → Time management and updates
- `stream` → Real-time updates (SSE)

### Responsibilities

- Receive parsed request data from routes
- Call into game state and services
- Aggregate results
- Return structured responses

### Example Flow

HTTP Request → Route → Controller → Game/Services → Response


Controllers do not store state—they orchestrate it.

---

### Game (`game/`)

The `game/` folder encapsulates all game-related state and logic.

#### Files

- `game_state` → Holds board state, FEN, move history
- `game_status` → Tracks game lifecycle (turn, checkmate, draw, etc.)

### Responsibilities

- Maintain authoritative game state
- Enforce turn order
- Detect game-ending conditions
- Provide clean interface for controllers

### Key Principle

This layer is **pure state + rules**, with no external dependencies.

---

### Routes (`routes/`)

Defines API endpoints exposed to the frontend.

#### Endpoints

- `/move` → Submit a move
- `/board` → Fetch board state
- `/engine` → Request evaluation
- `/clock` → Manage time
- `/stream` → Subscribe to updates (SSE)

### Responsibilities

- Map HTTP endpoints to controllers
- Perform lightweight validation
- Keep routing logic separate from business logic

---

### Services (`services/`)

Services act as the **bridge between the backend and external compute systems**.

#### Components

- Engine service → communicates with `engine_backend`
- PlayerBot service → communicates with `playerbot_backend`
- SSE service → streams updates to clients

### Responsibilities

- Handle outbound requests to Python services
- Normalize responses into backend-friendly format
- Manage communication protocols (HTTP / SSE)

### Key Insight

Services isolate external dependencies, allowing:
- Easier testing
- Cleaner controller logic
- Swappable backends

---

## Engine Backend (`engine_backend/`)

This service performs position evaluation using Stockfish.

### Responsibilities

- Accept FEN input
- Run engine analysis
- Return:
  - Best move
  - Top candidate lines
  - Evaluation scores

### Role in System

Provides **ground-truth evaluation** used by:
- Tutor system
- Move quality classification

---

## Player Bot Backend (`playerbot_backend/`)

This service generates human-like moves.

---

### HumanBot Design

Instead of relying on a single engine, the system uses **three distinct engines**:

1. **Weak Maia**
   - Simulates lower-skill human play
   - Produces more mistakes and variability

2. **Same-strength Maia**
   - Matches target ELO distribution
   - Provides realistic human-like decisions

3. **Same-strength Stockfish**
   - Provides tactical sharpness
   - Ensures strong moves appear in critical positions

---

### Policy-Based Engine Selection

A learned policy determines **which engine to use per move**.

#### Inputs to Policy

- Position complexity (e.g., tactical vs quiet)
- Player ELO setting
- Evaluation spread between candidate moves

#### Output

A probability distribution over the three engines:

P(engine | position, elo)

Example:
Weak Maia: 0.50
Same Maia: 0.30
Stockfish: 0.20



---

### Move Generation Process

1. Receive current FEN
2. Compute features (complexity, evaluation spread)
3. Use policy to sample engine
4. Query selected engine
5. Return chosen move

---

### Why This Works

- Weak Maia introduces realistic mistakes
- Strong engines handle tactical correctness
- Policy blending creates **human-like inconsistency**, which is critical for training

---

## End-to-End Request Flow

```js
Frontend
↓
API Route (/move)
↓
Controller (move)
↓
Game State Update
↓
Services Layer
↓
┌───────────────┬─────────────────┐
↓ ↓ ↓
Engine Backend PlayerBot SSE Stream
(Eval) (Move) (Update)
└───────────────┴─────────────────┘
↓
Controller aggregates response
↓
Frontend updates UI
```


---

## Real-Time Streaming (SSE)

The system uses Server-Sent Events (SSE) to push updates.

### Responsibilities

- Stream:
  - Board updates
  - Clock updates
  - Tutor feedback
- Keep clients synchronized without polling

### Benefit

- Lower latency
- Simpler than WebSockets for this use case
- Efficient one-way communication

---

## Key Takeaways

- `src/` is the authoritative backend and central coordinator
- Controllers orchestrate logic but do not store state
- Game layer owns all state and rules
- Services isolate external compute systems
- Engine backend provides optimal evaluation
- PlayerBot backend simulates human play using a policy over multiple engines

The system is modular, extensible, and designed to balance:
- Performance
- Realism
- Instructional value